# Example: Chen-Scott estimation


# Packages

Imported functions and modules are used for solving the exam.  

In [ ]:
import sys

import numpy as np
import pandas as pd
from matplotlib.lines import lineStyles

# statistical libraries
from scipy.stats import norm, multivariate_normal
from scipy import stats

# optimization functions
from scipy.optimize import minimize

# plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# typing
from typing import Union

## Helper functions

In [ ]:

def simulate_vasicek(initial_short_rate: float,
                     kappa: float,
                     theta: float,
                     sigma: float,
                     horizon: float,
                     dt: float = 1.0 / 52.0,
                     num_sim: int = 10000):
    """
    simulates short rate processes in a Vasicek setting until a given horizon

    Parameters
    ----------

    initial_short_rate:
        initial short rate
    kappa:
        speed of mean reversion.
    theta:
        long term mean of the short rate.
    sigma:
        volatility of the short rate.
    horizon:
        time until maturity/expiry (horizon).
    dt:
        increments in time
    num_sim:
        number of simulations.
    """
    std_rates = np.sqrt(sigma ** 2 / (2 * kappa) * (1 - np.exp(-2 * kappa * dt)))

    num_periods = int(horizon / dt)
    short_rates = np.empty((num_sim, num_periods))
    short_rates[:, 0] = initial_short_rate

    error_terms = np.random.normal(scale=std_rates, size=(num_sim, num_periods))

    for i in range(1, num_periods):
        short_rates[:, i] = theta + (short_rates[:, i - 1] - theta) * np.exp(-kappa * dt) + error_terms[:, i - 1]

    return short_rates


def calculate_zero_coupon_price(time_to_maturity: Union[float, np.ndarray],
                                initial_short_rate: float,
                                kappa: float,
                                theta: float,
                                sigma: float):

    """
    Computes the zero coupon yield for a given maturity and short rate using the Vasicek model.

    Parameters
    ----------
    time_to_maturity:
        time to maturity of the zero coupon bond.
    initial_short_rate:
        initial short rate.
    kappa:
        speed of mean reversion.
    theta:
        long term mean of the short rate under Q.
    sigma:
        volatility of the short rate.

    Returns
    -------
    zero_coupon_price:
        price of the zero coupon bond.
    """

    y_infty = theta - sigma ** 2 / (2 * kappa ** 2)

    b = 1 / kappa * (1 - np.exp(-kappa * time_to_maturity))
    a = y_infty * (time_to_maturity - b) + sigma ** 2 / (4 * kappa) * b ** 2

    return np.exp(- a - b * initial_short_rate)

def calculate_zero_coupon_yield(time_to_maturity: Union[float, np.ndarray],
                                initial_short_rate: float,
                                kappa: float,
                                theta: float,
                                sigma: float):

    """
    Computes the zero coupon yield for a given maturity and short rate using the Vasicek model.

    Parameters
    ----------
    time_to_maturity:
        time to maturity of the zero coupon bond.
    initial_short_rate:
        initial short rate.
    kappa:
        speed of mean reversion.
    theta:
        long term mean of the short rate under Q.
    sigma:
        volatility of the short rate.

    Returns
    -------
    zero_coupon_yield:
        yield of the zero coupon bond.
    """


    price = calculate_zero_coupon_price(time_to_maturity=time_to_maturity,
                                        initial_short_rate=initial_short_rate,
                                        kappa=kappa,
                                        theta=theta,
                                        sigma=sigma)

    return - np.log(price) / time_to_maturity


## Simulate yield curve data

In [ ]:
# set random seed for reproducibility
np.random.seed(55) # 55

np.random.seed(5)

"""
General settings
"""

dt = 1.0 / 52.0 # time step in years (weekly)
sample_length = 40 # total sample length in years
num_periods = int(sample_length / dt) # number of time periods
time_steps = np.linspace(0, sample_length, num_periods) # time grid

"""
Fixed income data 

    1. generate data for short rate using Vasicek model
    2. generate zero-coupon bond yields with measurement error 
    3. generate bullet bonds (price for a given maturity and coupon rate) using the provided yield curve
    
"""


# define true parameters
init_short_rate = 0.03  # initial short rate
kappa = 0.04         # speed of mean reversion
theta_q = 0.129         # long-term mean level under q
sigma = 0.0175          # volatility
lam = -0.0035          # market price of risk
theta_p = theta_q + lam / kappa

# simulate short rate under physical measure
short_rates = simulate_vasicek(initial_short_rate=init_short_rate,
                               kappa=kappa,
                               theta=theta_p,
                               sigma=sigma,
                               horizon=sample_length,
                               dt=dt,
                               num_sim=1).flatten()

# simulate zero coupon bond yields with measurement error
tenors = np.array([0.25, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])  # in years
df_zcb_yields = pd.DataFrame(data=np.zeros((len(short_rates) , len(tenors))), columns=tenors, index=time_steps)
df_discount_factors = pd.DataFrame(data=np.zeros((len(short_rates) , len(tenors))), columns=tenors, index=time_steps)
for i, t in enumerate(time_steps):

    # calculate "true" zero coupon yields
    zcb_yields = calculate_zero_coupon_yield(time_to_maturity=tenors,
                                             initial_short_rate=short_rates[i],
                                             kappa=kappa,
                                             theta=theta_q,
                                             sigma=sigma)

    # add measurement error
    zcb_yields[1:] += np.random.normal(loc=0.0, scale=0.002, size=len(tenors)-1)

    # store in dataframe
    df_zcb_yields.iloc[i, :] = zcb_yields
    df_discount_factors.iloc[i, :] = np.exp(- zcb_yields * tenors)

# price bullet bonds using the generated yield curve
# assume 6% coupon rate, annual coupon payments, maturity 1 to 10 years
bullet_bond_cash_flow_matrix = np.tri(10) * 0.06 + np.eye(10) * 1
df_bullet_bond_prices = df_discount_factors.iloc[:, 1:].values @ bullet_bond_cash_flow_matrix.T

# define output dataframe
df_fixed_income = pd.DataFrame(data=None,
                                     columns= ['3M ZCB'] + [f'Bullet_Bond_{i+1}Y' for i in range(10)],
                                     index=time_steps)

df_fixed_income.iloc[:, 1:] = df_bullet_bond_prices
df_fixed_income.iloc[:, 0] = df_discount_factors.iloc[:, 0]


In [ ]:
df_fixed_income.to_csv("fixed_income_data.csv")

## Plot and bootstrap zero coupon yield curve

It is assumed that we observed the bullet bond prices and NOT the true latent short rate!

In [ ]:
df_fixed_income = pd.read_csv('fixed_income_data.csv', index_col=0)


df_fixed_income["3M ZCB"].plot()

In [ ]:
"""
build cash flow matrix for bootstrapping zero-coupon curve
"""

cash_flow_time_points = np.array([0.25] + list(range(1, 11)))

# allocate cash flow matrix
cash_flow_matrix = np.zeros((11, 11))

# fill element corresponding to 3M ZCB
cash_flow_matrix[0, 0] = 1

# coupon rate for bullet bonds
coupon_rate = 0.06

# fill elements corresponding to bullet bonds
bullet_bond_cash_flow_matrix = np.tri(10) * coupon_rate * 1 + np.eye(10) * 1
cash_flow_matrix[1:11, 1:11] = bullet_bond_cash_flow_matrix

In [ ]:
cash_flow_matrix

In [ ]:
"""
Bootstrap curve for each point in time
"""

# obtain discount factors by bootstrapping
df_discount_factors = (np.linalg.inv(cash_flow_matrix) @ df_fixed_income.T).T
df_discount_factors.columns = cash_flow_time_points

# obtain cont. compounded zero rates
df_zero_rates = -np.log(df_discount_factors) / cash_flow_time_points

In [ ]:
df_discount_factors.head()

In [ ]:
df_zero_rates.head()

In [ ]:
df_zero_rates.iloc[75, :].plot()

### Important observations regarding the data generating process

Yields are almost perfectly correlated across maturities, which indicates that one factor drives the yield curve dynamics!

In [ ]:
"""
Correlation matrix of zero rates
"""

# calculate correlation matrix
corr_matrix_zero_rates = df_zero_rates.corr()

# plot correlation matrix
# plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix_zero_rates, annot=True, fmt=".2f", cmap='coolwarm', xticklabels=cash_flow_time_points, yticklabels=cash_flow_time_points)
plt.xlabel('Maturity (Years)');
plt.ylabel('Maturity (Years)');
plt.title('Correlation Matrix of Zero Rates');

Yield changes appears to be normally distributed, which is consistent with the Vasicek model assumption of normally distributed interest rates!

In [ ]:
"""
Plot density of changes in 5Y zero rates
"""

zero_rates_to_plot = df_zero_rates.iloc[:, 5]
change_zero_rates_to_plot = zero_rates_to_plot.diff().dropna()

plt.figure(figsize=(10, 6))
# plot density of zero rates
sns.kdeplot(change_zero_rates_to_plot, fill=True)
# add normal fit
mu, std = norm.fit(change_zero_rates_to_plot)
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, std)
plt.plot(x, p, 'k', linewidth=2, label='Normal Fit')
plt.title('Density of Changes in 5Y Zero Rate')
plt.xlabel('Change in Zero Rate')
plt.ylabel('Density')
plt.legend()



### Estimating dynamics of the yield curve

It is possible to model the yield curve using different approaches. The underlying data is generated using a Vacicek model with measurement error added to the observed zero rates, but not the 3M zero rate.

I apply a Chen-Scott approach to estimate the model parameters in a one-factor Vasicek model with i.i.d. and constant measurement error.

In [ ]:
"""
Helper functions
"""


def extract_state_variable(time_to_maturity: Union[float, np.ndarray],
                           y: float,
                           kappa: float,
                           theta: float,
                           sigma: float) -> Union[float, np.ndarray]:

    """
    Extract the latent state variable (short rate) from the zero coupon yield using the Vasicek model.

    Parameters
    ----------
    time_to_maturity:
        maturity of the zero coupon bond
    y:
        zero coupon yield
    kappa:
        speed of mean reversion
    theta:
        long term mean of the short rate under Q
    sigma:
        volatility of the short rate

    Returns
    -------
    short rate (latent state variable)

    """

    # convergence level of the zero coupon yield
    y_infty = theta - sigma ** 2 / (2 * kappa ** 2)

    # compute the coefficients a and b for the zero coupon yield
    b = 1 / kappa * (1 - np.exp(-kappa * time_to_maturity))
    a = y_infty * (time_to_maturity - b) + sigma ** 2 / (4 * kappa) * b ** 2

    return (time_to_maturity * y - a) / b


def vasicek_log_unconditional_density(short_rate: Union[float, np.ndarray],
                                      kappa: float,
                                      theta: float,
                                      sigma: float) -> Union[float, np.ndarray]:

    """
    Compute the log unconditional density of the short rate in the Vasicek model.

    Parameters
    ----------
    short_rate:
        short rate(s)
        Can handle multiple time-points (np.ndarray) or a single time-point (float)
    kappa:
        speed of mean reversion
    theta:
        long term mean of the short rate
    sigma:
        volatility of the short rate

    Returns
    -------
    Union[float, np.ndarray]
        Log unconditional density of the short rate

    """

    # standard deviation of short rate
    sigma_r = np.sqrt(sigma ** 2 / (2 * kappa))

    # mean of short rate
    mu_r = theta

    return norm(loc=mu_r, scale=sigma_r).logpdf(short_rate)


def vasicek_log_conditional_density(short_rate: Union[float, np.ndarray],
                                    short_rate_lagged: Union[float, np.ndarray],
                                    kappa: float,
                                    theta: float,
                                    sigma: float,
                                    dt: float = 1.0 / 52.0) -> Union[float, np.ndarray]:

    """
    Compute the log conditional density of the short rate in the Vasicek model.

    Parameters
    ----------
    short_rate:
        short rate
    short_rate_lagged:
        lagged short rate (previous time period)
    kappa:
        speed of mean reversion
    theta:
        long term mean of the short rate
    sigma:
        volatility of the short rate
    dt:
        time step size (default is 1 week)

    Returns
    -------
    Union[float, np.ndarray]
        Log conditional density of the short rate

    """

    # standard deviation of short rate
    sigma_r = np.sqrt(sigma ** 2 / (2 * kappa) * (1 - np.exp(-2 * kappa * dt)))

    # mean of short rate
    mu_r = theta + (short_rate_lagged - theta) * np.exp(-kappa * dt)

    return norm(loc=mu_r, scale=sigma_r).logpdf(short_rate)


def neg_vasicek_log_likelihood(params: np.ndarray,
                               short_rates: np.ndarray,
                               individual_contribution: bool = False,
                               dt: float = 1.0 / 52.0) -> Union[float, np.ndarray]:

    """
    Compute the negative log-likelihood of the Vasicek model.

    Parameters
    ----------
    params:
        Parameters of the Vasicek model: [kappa, theta, sigma]
    short_rates:
        Short rate time series
    individual_contribution:
        If True, return individual log-likelihood contributions, otherwise return the mean log-likelihood
    dt:
        Time step size (default is 1 week)

    Returns
    -------
    Union[float, np.ndarray]
        Negative log-likelihood of the Vasicek model

    """

    kappa = params[0]
    theta = params[1]
    sigma = params[2]

    log_likes = np.empty_like(short_rates)
    log_likes[0] = -vasicek_log_unconditional_density(short_rates[0], kappa, theta, sigma)
    log_likes[1:] = -vasicek_log_conditional_density(short_rates[1:], short_rates[:-1], kappa, theta, sigma, dt=dt)

    if individual_contribution:
        return log_likes
    else:
        return log_likes.mean()


def neg_chen_scott_log_likelihood(params: np.ndarray,
                                  zero_yields: np.ndarray,
                                  time_to_maturities: np.ndarray,
                                  individual_contribution: bool=False,
                                  dt: float = 1.0 / 52.0) -> Union[float, np.ndarray]:

    """
    Compute the negative log-likelihood used for Chen-Scott estimation of the Vasicek model using both P and Q dynamics.

    Parameters
    ----------
    params:
        Parameters of the Vasicek model: [gamma, m, sigma, lam, sigma_eps]
    zero_yields:
        Zero coupon yields time series, shape (k_bar, t_bar)
    time_to_maturities:
        Maturities of the zero coupon bonds, shape (k_bar,)
    individual_contribution:
        If True, return individual log-likelihood contributions, otherwise return the mean log-likelihood
    dt:
        Time step size (default is 1 week)

    Returns
    -------
    Union[float, np.ndarray]
        Negative log-likelihood
    """

    # number of time periods and number of zero coupon yields
    k_bar, t_bar = zero_yields.shape

    # initialize parameters
    kappa = params[0]
    theta_q = params[1]
    sigma = params[2]
    theta_p = params[3]
    sigma_eps = np.ones(k_bar - 1) * params[4] # params[4:]

    # extract latent state variable - assume the shortest yield is measured without error
    short_rates = extract_state_variable(time_to_maturities[0], zero_yields[0], kappa, theta_q, sigma)

    # compute the log-likelihood of the state variable
    log_likes = neg_vasicek_log_likelihood(np.array([kappa, theta_p, sigma]), short_rates, individual_contribution=True, dt=dt)

    # add the Jacobian term for the yield-to-state transformation
    b = 1 / kappa * (1 - np.exp(-kappa * time_to_maturities)) * 1 / time_to_maturities
    b_mat = np.eye(k_bar)
    b_mat[:, 0] = b
    sign, logdet_B = np.linalg.slogdet(b_mat)

    # or np.log(b[0])

    if sign <= 0:
        raise ValueError("Jacobian matrix B must have a positive determinant.")
    log_likes += logdet_B

    # compute the measurement error
    eps = zero_yields[1:].T - np.array([calculate_zero_coupon_yield(time_to_maturities[1:], _, kappa, theta_q, sigma) for _ in short_rates])

    # add measurement error to the log-likelihood
    log_likes += -multivariate_normal.logpdf(eps, mean=np.zeros_like(sigma_eps), cov=np.diag(sigma_eps**2))

    # return the log-likelihood of the state variable and measurement error
    if individual_contribution:
        return log_likes
    else:
        return log_likes.mean() 



In [ ]:
k_bar = 11

b = 1 / kappa * (1 - np.exp(-kappa * time_to_maturities)) * 1 / time_to_maturities
b_mat = np.eye(k_bar)
b_mat[:, 0] = b
sign, logdet_B = np.linalg.slogdet(b_mat)

print(np.log(b[0]))
print(logdet_B)

In [ ]:
#idx = [0, 1, 5, 7, 10]

"""
Estimate parameters using Chen-Scott approach
"""

# initial parameter guess
initial_params = np.array([0.05, 0.11, 0.02, 0.05, 0.0025])
# bounds for the parameters
param_bounds = [(1e-5, None), (1e-5, None), (1e-5, None), (1e-5, None), (1e-5, None)]
# optimization
result = minimize(neg_chen_scott_log_likelihood,
                  initial_params,
                  args=(df_zero_rates.T.values, cash_flow_time_points),
                  bounds=param_bounds,
                  method='nelder-mead',
                  options={'maxiter': 1000,
                           'disp': True})

# estimated parameters
estimated_params = result.x

print("Estimated Parameters:")
print(f"kappa: {estimated_params[0]:.4f}")
print(f"theta_q: {estimated_params[1]:.4f}")
print(f"sigma: {estimated_params[2]:.4f}")
print(f"theta_p: {estimated_params[3]:.4f}")
print(f"sigma_eps: {estimated_params[4]:.4f}")

In [ ]:
theta_q

In [ ]:
"""
Allocate parameters
"""

kappa_est = estimated_params[0]
theta_q_est = estimated_params[1]
sigma_est = estimated_params[2]
theta_p_est = estimated_params[3]
sigma_eps_est = estimated_params[4]

In [ ]:
kappa_est

In [ ]:
theta_q_est

In [ ]:
theta_p_est

In [ ]:
"""
Calculate fitted zero rates
"""

time_to_maturities = cash_flow_time_points
num_per = df_zero_rates.shape[0]

short_rates = np.array([extract_state_variable(time_to_maturities[0],
                                     df_zero_rates.iloc[t, 0],
                                     kappa_est,
                                     theta_q_est,
                                     sigma_est)
                        for t in range(num_per)])

fitted_zero_rates = np.array([calculate_zero_coupon_yield(time_to_maturities,
                                                          short_rates[t],
                                                          kappa_est,
                                                          theta_q_est,
                                                          sigma_est)
                              for t in range(num_per)])

In [ ]:
"""
Plot fitted vs. observed zero rates
"""

tenor = 10 # 10-year zero rate
plt.figure(figsize=(10, 6))
plt.plot(df_zero_rates.index, df_zero_rates[tenor], label=f'Observed {tenor}Y Zero Rate', color='blue')
plt.plot(df_zero_rates.index, fitted_zero_rates[:, np.where(time_to_maturities == tenor)[0][0]], label=f'Fitted {tenor}Y Zero Rate', color='orange', linestyle='--')
plt.title('Observed vs Fitted 10-Year Zero Rates')
plt.xlabel('Time')
plt.ylabel('Zero Rate')
plt.legend()
plt.grid()

plt.savefig(f'observed_vs_fitted_{tenor}Y_zero_rates.png')

plt.show()


In [ ]:
"""
Plot the evolution of the short rate
"""

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_zero_rates.index, short_rates)
ax.set_xlabel('Time')
ax.set_ylabel('Short Rate')
ax.set_title('Short Rate')
plt.grid()
